In [1]:
# ============================================
# MLflow Manual Logging - Linear Regression
# Dataset: California Housing (sklearn built-in)
# ============================================

import logging
import warnings
import numpy as np
import pandas as pd
from urllib.parse import urlparse

from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature

logging.basicConfig(level=logging.WARN)
logger = logging.getLogger(__name__)
warnings.filterwarnings("ignore")
np.random.seed(40)

# --- Helper function ---
def eval_metrics(actual, pred):
    rmse = np.sqrt(mean_squared_error(actual, pred))
    mae = mean_absolute_error(actual, pred)
    r2 = r2_score(actual, pred)
    return rmse, mae, r2

# --- Load California Housing dataset ---
housing = fetch_california_housing(as_frame=True)
data = housing.frame
print(f"Dataset shape: {data.shape}")
print(data.head())

# Split the data into training and test sets (0.75, 0.25)
train, test = train_test_split(data)

# Target column is "MedHouseVal" (median house value)
train_x = train.drop(["MedHouseVal"], axis=1)
test_x = test.drop(["MedHouseVal"], axis=1)
train_y = train[["MedHouseVal"]]
test_y = test[["MedHouseVal"]]

alpha = 0.5
l1_ratio = 0.5

# --- MLflow Run ---
with mlflow.start_run():
    lr = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, random_state=42)
    lr.fit(train_x, train_y)

    predicted = lr.predict(test_x)

    (rmse, mae, r2) = eval_metrics(test_y, predicted)

    print(f"ElasticNet model (alpha={alpha:.4f}, l1_ratio={l1_ratio:.4f}):")
    print(f"  RMSE: {rmse}")
    print(f"  MAE: {mae}")
    print(f"  R2: {r2}")

    # Log params and metrics
    mlflow.log_param("alpha", alpha)
    mlflow.log_param("l1_ratio", l1_ratio)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)
    mlflow.log_metric("mae", mae)

    # Log model
    predictions = lr.predict(train_x)
    signature = infer_signature(train_x, predictions)

    tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

    if tracking_url_type_store != "file":
        mlflow.sklearn.log_model(
            lr, "model", registered_model_name="ElasticnetHousingModel", signature=signature
        )
    else:
        mlflow.sklearn.log_model(lr, "model", signature=signature)

print("\nModel logged successfully!")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

c:\Users\nishi\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset shape: (20640, 9)
   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   

   Longitude  MedHouseVal  
0    -122.23        4.526  
1    -122.22        3.585  
2    -122.24        3.521  
3    -122.25        3.413  
4    -122.25        3.422  


2026/03/31 22:13:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


ElasticNet model (alpha=0.5000, l1_ratio=0.5000):
  RMSE: 0.8405383940492156
  MAE: 0.6350501830593794
  R2: 0.4804248608261792


2026/03/31 22:13:35 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



Model logged successfully!
Tracking URI: sqlite:///C:/Users/nishi/MLOPS_LAB05/Experiment_Tracking_Labs/Mlflow_Labs/Lab1/mlflow.db


Successfully registered model 'ElasticnetHousingModel'.
Created version '1' of model 'ElasticnetHousingModel'.
